# 04 Travel times

This notebook builds the travel time matrix: how many minutes it takes to drive from each of the 3,465 origins to each of the 13 attractions. The model in 05 uses these times for distance decay (people visit far places less often).

**Snapping.** Origins and attractions are points with coordinates, but a route can only start and end at a network node (a road junction or road end). Snapping moves every point to its nearest node. The network has major roads only, so a village can be a few km from its nearest node; that gap is added as a short access time on local roads.

**Dijkstra's algorithm.** It finds the fastest routes from one start node to every other node in a single run, by always continuing from the closest node it has not finished yet. So one run per attraction (13 runs) gives all 3,465 × 13 times, instead of 45,045 separate route searches.

**Reversed graph.** Dijkstra measures times *from* its start node, but we need times from the origins *to* an attraction. Running it on a copy of the network where every road points the other way gives exactly those times, and one-way roads (ramps, divided highways) are still respected.

Times use the calibrated edge weight `travel_time` from 03 (weekend morning, slow zone around St Petersburg). Output: `data/processed/travel_times.csv`.

In [16]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

RAW = "../data/raw"
PROCESSED = "../data/processed"
CRS_M = "EPSG:32636"  # UTM zone 36N, metres

# travel_time is converted to numbers automatically; our own travel_time_ff column needs telling
G = ox.load_graphml(os.path.join(RAW, "roads_drive.graphml"), edge_dtypes={"travel_time_ff": float})
origins = pd.read_csv(os.path.join(PROCESSED, "origins.csv"))
att = pd.read_csv(os.path.join(PROCESSED, "attractions_geo.csv"))

tt = [d["travel_time"] for _, _, d in G.edges(data=True)]
print(f"Network: {len(G):,} nodes, {G.number_of_edges():,} edges, CRS {G.graph['crs']}")
print(f"travel_time is numeric: {isinstance(tt[0], float)}, mean {np.mean(tt):.1f} s per edge")
print(f"Origins: {len(origins):,}, population {origins['population'].sum():,.0f}")
print(f"Attractions: {len(att)}", att["role"].value_counts().to_dict())

Network: 9,789 nodes, 20,074 edges, CRS epsg:4326
travel_time is numeric: True, mean 114.0 s per edge
Origins: 3,465, population 8,265,799
Attractions: 13 {'calibration': 9, 'context': 3, 'competitor': 1}


In [17]:
# Snap every origin and attraction to its nearest network node
# (graph is in lat/lon, so osmnx uses scikit-learn and returns distances in metres)
origins["node"], origins["snap_m"] = ox.distance.nearest_nodes(
    G, X=origins["lon"], Y=origins["lat"], return_dist=True)
att["node"], att["snap_m"] = ox.distance.nearest_nodes(
    G, X=att["lon"], Y=att["lat"], return_dist=True)

# Snap distance summary for origins
km = origins["snap_m"] / 1000
print("Origin snap distance, km:")
print(km.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())
for limit in [2, 5, 10]:
    far = origins[km > limit]
    print(f"Origins > {limit} km from a node: {len(far):,} "
          f"({far['population'].sum() / origins['population'].sum():.1%} of population)")
print(f"Distinct nodes used by origins: {origins['node'].nunique():,}")

print("\n10 farthest origins:")
print(origins.nlargest(10, "snap_m")[["origin_id", "region", "population", "snap_m"]]
      .round(0).to_string(index=False))

print("\nAttractions:")
print(att[["name_en", "role", "node", "snap_m"]].round(0).to_string(index=False))

Origin snap distance, km:
count    3465.00
mean        5.73
std         6.20
min         0.02
50%         4.19
90%        12.64
99%        25.64
max        90.17
Origins > 2 km from a node: 2,430 (8.4% of population)
Origins > 5 km from a node: 1,495 (1.8% of population)
Origins > 10 km from a node: 580 (0.5% of population)
Distinct nodes used by origins: 1,385

10 farthest origins:
 origin_id           region  population  snap_m
      2665 Leningrad Oblast        12.0 90167.0
      2377 Leningrad Oblast        28.0 79025.0
      1997 Leningrad Oblast        65.0 78468.0
      1613 Leningrad Oblast       121.0 77556.0
      1339 Leningrad Oblast       186.0 76807.0
      3137 Leningrad Oblast         2.0 76337.0
      2032 Leningrad Oblast        61.0 59356.0
      3023 Leningrad Oblast         4.0 46526.0
      2309 Leningrad Oblast        35.0 46261.0
      3210 Leningrad Oblast         2.0 45860.0

Attractions:
                                  name_en        role       node  snap_m

## Access time and unreachable origins

The snapping check found 25 origins more than 30 km from the network (788 people): small islands in the Gulf of Finland and Lake Ladoga, and a roadless forest area in the far east of the oblast near the Vologda border. They cannot reach the attractions by car along this network and are dropped.

For origins, the gap between the origin centre and its node gets an access time: straight-line distance × 1.3 (real roads wind) at 40 km/h (local roads). Attractions get no access time. The Yandex check below showed that the network time to an attraction's node already matches Yandex (Izvara 117 vs 116 min), while adding access time made Izvara and Staraya Ladoga 16-20 min too slow, so it counted the last stretch twice.

In [18]:
MAX_SNAP_KM = 30   # farther than this = island or roadless forest, not reachable by car
ACCESS_KMH = 40    # average speed on local roads between an origin and the major network
DETOUR = 1.3       # real roads are longer than the straight line

dropped = origins[origins["snap_m"] > MAX_SNAP_KM * 1000]
origins = origins[origins["snap_m"] <= MAX_SNAP_KM * 1000].reset_index(drop=True)
print(f"Dropped {len(dropped)} origins ({dropped['population'].sum():,.0f} people), "
      f"{len(origins):,} origins left")

def access_min(snap_m):
    return snap_m / 1000 * DETOUR / ACCESS_KMH * 60

origins["access_min"] = access_min(origins["snap_m"])
att["access_min"] = 0.0  # network time to the node already matches Yandex (see sanity checks)

print("\nOrigin access time, min:")
print(origins["access_min"].describe(percentiles=[0.5, 0.9]).round(1).to_string())

Dropped 25 origins (788 people), 3,440 origins left

Origin access time, min:
count    3440.0
mean       10.6
std         9.5
min         0.0
50%         8.1
90%        24.2
max        57.9


In [19]:
# Reversed graph: every road points the other way, so Dijkstra FROM an attraction
# gives the fastest times TO that attraction from every node (one-way roads respected)
G_rev = G.reverse(copy=False)

times = {}
for _, a in att.iterrows():
    sec = nx.single_source_dijkstra_path_length(G_rev, a["node"], weight="travel_time")
    network_min = origins["node"].map(sec) / 60
    times[a["name_en"]] = network_min + origins["access_min"] + a["access_min"]

tt = pd.DataFrame(times)
tt.index = origins["origin_id"]

print(f"Matrix: {tt.shape[0]:,} origins × {tt.shape[1]} attractions, missing values: {tt.isna().sum().sum()}")
print("\nTravel time per attraction, minutes:")
print(tt.describe().T[["min", "50%", "max"]].round(0).to_string())

Matrix: 3,440 origins × 13 attractions, missing values: 0

Travel time per attraction, minutes:
                                            min    50%    max
Slantsy Local History Museum                1.0  250.0  513.0
Kingisepp Local History Museum              2.0  216.0  469.0
Ivangorod Museum (fortress and galleries)   1.0  231.0  485.0
Koporye Fortress                            1.0  199.0  449.0
Staraya Ladoga Museum-Reserve               1.0  184.0  377.0
Sablino caves and waterfalls               10.0  184.0  367.0
Putilovo Paleopark (fossil digging)         5.0  175.0  354.0
Gdov Kremlin and Local Museum               5.0  287.0  554.0
Oreshek Fortress (Shlisselburg)             4.0  178.0  384.0
Roerich Estate Museum (Izvara)              2.0  181.0  428.0
Yam Fortress ramparts (Kingisepp)           2.0  216.0  469.0
Lopukhinka radon springs                    1.0  185.0  427.0
Lava river canyon                           4.0  180.0  349.0


## Sanity checks

**Yandex comparison.** Model times for seven routes (car, Saturday 10:00 for routes from St Petersburg) are all within 8 min of Yandex Maps. The largest gaps are Putilovo (model 8 min faster) and Staraya Ladoga (model takes a longer route, see 03).

**Population view.** For the average resident (population-weighted median), Oreshek is the closest attraction (83 min) and Slantsy one of the farthest (about 3 hours); only about 4% of the study area population lives within 2 hours of Slantsy. This is the starting point of the Slantsy question: the museum is far from where people live, so distance decay will weigh heavily on it.

The matrix is saved as `data/processed/travel_times.csv` (one row per origin, one column per attraction, minutes). The map shows the travel time from every origin to Slantsy.

In [20]:
# Model vs Yandex (car; SPb routes Sat 7 Nov 10:00, rural routes checked 5 Nov)
# Yandex routes end at town/village centres or the museum itself, so a few minutes' difference is expected
START = {"Palace Square": (59.9391, 30.3158), "Slantsy": (59.118, 28.088),
         "Kingisepp": (59.373, 28.613)}
CHECKS = [  # (start, attraction, Yandex minutes)
    ("Palace Square", "Slantsy Local History Museum", 187),
    ("Palace Square", "Ivangorod Museum (fortress and galleries)", 161),
    ("Palace Square", "Staraya Ladoga Museum-Reserve", 133),
    ("Palace Square", "Roerich Estate Museum (Izvara)", 116),
    ("Palace Square", "Putilovo Paleopark (fossil digging)", 106),
    ("Kingisepp", "Slantsy Local History Museum", 48),
    ("Slantsy", "Gdov Kremlin and Local Museum", 38),
]

att_idx = att.set_index("name_en")
rows = []
for start, name, yandex in CHECKS:
    lat, lon = START[start]
    s_node, s_snap = ox.distance.nearest_nodes(G, X=lon, Y=lat, return_dist=True)
    a = att_idx.loc[name]
    net = nx.shortest_path_length(G, s_node, a["node"], weight="travel_time") / 60
    rows.append({"route": f"{start} -> {name[:25]}", "network": net,
                 "access": access_min(s_snap) + a["access_min"], "yandex": yandex})

check = pd.DataFrame(rows)
check["model"] = check["network"] + check["access"]
check["diff"] = check["model"] - check["yandex"]
print(check.round(0).to_string(index=False))

                                     route  network  access  yandex  model  diff
Palace Square -> Slantsy Local History Mus    188.0     0.0     187  188.0   1.0
Palace Square -> Ivangorod Museum (fortres    159.0     0.0     161  160.0  -1.0
Palace Square -> Staraya Ladoga Museum-Res    139.0     0.0     133  140.0   7.0
Palace Square -> Roerich Estate Museum (Iz    117.0     0.0     116  117.0   1.0
Palace Square -> Putilovo Paleopark (fossi     98.0     0.0     106   99.0  -7.0
    Kingisepp -> Slantsy Local History Mus     46.0     0.0      48   47.0  -1.0
      Slantsy -> Gdov Kremlin and Local Mu     41.0     0.0      38   41.0   3.0


In [21]:
# Population-weighted view: how far are attractions for the average resident?
pop = origins.set_index("origin_id")["population"]

def weighted_median(s):
    s = s.sort_values()
    cum = pop[s.index].cumsum()
    return s[cum >= pop.sum() / 2].iloc[0]

summary = pd.DataFrame({
    "median_min_people": tt.apply(weighted_median),
    "pop_within_60": tt.apply(lambda s: pop[s <= 60].sum() / pop.sum()),
    "pop_within_120": tt.apply(lambda s: pop[s <= 120].sum() / pop.sum()),
})
# Which attraction is closest for each origin, as a share of all people
nearest = tt.idxmin(axis=1)
summary["nearest_for"] = pop.groupby(nearest).sum() / pop.sum()
summary = summary.fillna(0).sort_values("median_min_people")

print(summary.round({"median_min_people": 0, "pop_within_60": 3,
                     "pop_within_120": 3, "nearest_for": 3}).to_string())

                                           median_min_people  pop_within_60  pop_within_120  nearest_for
Oreshek Fortress (Shlisselburg)                         82.0          0.077           0.810        0.697
Putilovo Paleopark (fossil digging)                     92.0          0.058           0.819        0.015
Lopukhinka radon springs                                97.0          0.041           0.779        0.047
Lava river canyon                                      100.0          0.047           0.802        0.001
Sablino caves and waterfalls                           102.0          0.042           0.809        0.092
Roerich Estate Museum (Izvara)                         115.0          0.048           0.623        0.058
Koporye Fortress                                       118.0          0.032           0.633        0.013
Staraya Ladoga Museum-Reserve                          131.0          0.028           0.223        0.055
Kingisepp Local History Museum                         

In [22]:
# Save the matrix: one row per origin, one column per attraction, minutes
out = tt.round(1).reset_index()
out.to_csv(os.path.join(PROCESSED, "travel_times.csv"), index=False)
print(f"Saved travel_times.csv: {out.shape[0]:,} rows × {out.shape[1]} columns "
      f"({os.path.getsize(os.path.join(PROCESSED, 'travel_times.csv')) / 1024:.0f} KB)")

Saved travel_times.csv: 3,440 rows × 14 columns (276 KB)


In [23]:
# Map: travel time to Slantsy from every origin
import folium

SLANTSY = "Slantsy Local History Museum"
EDGES = [0, 60, 120, 180, 240, np.inf]
LABELS = ["under 1 h", "1-2 h", "2-3 h", "3-4 h", "over 4 h"]
COLORS = ["#1b3a6b", "#4393c3", "#92c5de", "#f4a582", "#d6604d"]

t = tt[SLANTSY]
band = pd.cut(t, EDGES, labels=LABELS)
print("Share of population by travel time to Slantsy:")
print((pop.groupby(band, observed=False).sum() / pop.sum()).round(3).to_string())

m = folium.Map(location=[59.5, 30.5], zoom_start=7, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, © OpenStreetMap contributors", name="Light gray").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, © OpenStreetMap contributors", name="Streets").add_to(m)

# Origins: colour = time band, size = population (small dots first so big ones sit on top)
o = origins.assign(minutes=t.values, band=band.values).sort_values("population")
for _, r in o.iterrows():
    color = COLORS[LABELS.index(r["band"])]
    folium.CircleMarker(
        [r["lat"], r["lon"]], radius=2 + 1.5 * np.log10(r["population"] + 1),
        color=color, weight=0, fill=True, fill_color=color, fill_opacity=0.8,
        tooltip=f"{r['minutes']:.0f} min to Slantsy | {r['population']:,.0f} people | {r['region']}",
    ).add_to(m)

# Attractions: Slantsy as a star, the others as small dark markers
for _, a in att.iterrows():
    if a["name_en"] == SLANTSY:
        folium.Marker([a["lat"], a["lon"]], tooltip=a["name_en"],
                      icon=folium.Icon(color="red", icon="star")).add_to(m)
    else:
        folium.CircleMarker([a["lat"], a["lon"]], radius=5, color="#333333", weight=2,
                            fill=True, fill_color="white", fill_opacity=1,
                            tooltip=f"{a['name_en']} ({a['role']})").add_to(m)

legend = "".join(f'<div><span style="background:{c};width:12px;height:12px;display:inline-block;'
                 f'margin-right:6px;border-radius:50%"></span>{l}</div>' for c, l in zip(COLORS, LABELS))
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;top:12px;right:12px;z-index:9999;background:white;padding:8px 10px;'
    f'border-radius:6px;font:12px sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)">'
    f'<b>Drive time to Slantsy</b>{legend}</div>'))
folium.LayerControl(position="bottomright").add_to(m)
m

Share of population by travel time to Slantsy:
Slantsy Local History Museum
under 1 h    0.014
1-2 h        0.030
2-3 h        0.283
3-4 h        0.574
over 4 h     0.099
